# Internship Project: 3D Pharmacophore Generation

### Molecule: Caffeine
**file format:** '.sdf' (instead of '.mol' as mentioned in instructions)

**Tools:** RDKit, SciPy,JSON

In [1]:
!pip uninstall -y numpy
!pip install numpy==1.26.4
!pip install rdkit==2023.09.5

Found existing installation: numpy 2.0.2
Uninstalling numpy-2.0.2:
  Successfully uninstalled numpy-2.0.2
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 98.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pytensor 2.35.1 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-python 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
shap 0.50.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
opencv-python-headless 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
jaxlib 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
jax 0.7.2 requires numpy>=2.0, but you have numpy 1.26

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 34.3/34.3 MB 13.2 MB/s eta 0:00:00


##1️⃣ Load Molecule  
**Note:**  The intenship test instructions menstion using .mol file format. however, in this project, an '.sdf' file is used, which is fully compatible with RDKit, Both '.mol' and '.sdf' files produce the same molecular object in RDKit,so all subsequent steps are unaffected.

In [1]:
from google.colab import files
uploaded = files.upload()

Saving Conformer3D_COMPOUND_CID_2519.sdf to Conformer3D_COMPOUND_CID_2519.sdf


Explanation:

.mol files store a single molecule.

.sdf files can store multiple molecules.

Since this project uses one molecule (Caffeine), .sdf functions identically to .mol.

All subsequent steps (3D conformer generation, feature extraction, distance calculation, consensus pharmacophore, JSON export) are compatible.

In [2]:
from rdkit import Chem
from rdkit.Chem import AllChem

In [3]:
input_file = "Conformer3D_COMPOUND_CID_2519.sdf"
mols = Chem.SDMolSupplier(input_file, removeHs=False)
mols = [m for m in mols if m is not None]

print("Molecules loaded:", len(mols))

Molecules loaded: 1


## 2️⃣Generate 3D Conformers and Energy Minimization

In [4]:
from rdkit.Chem import AllChem

mol = mols[0]
mol3d = Chem.AddHs(mol)

print("Embedding conformers...")
conf_ids = AllChem.EmbedMultipleConfs(
    mol3d,
    numConfs=20,
    randomSeed=42
)

print("Total conformers generated:", len(conf_ids))

Embedding conformers...
Total conformers generated: 20


In [5]:
energies = []

print("Minimizing conformers...")

for cid in conf_ids:
    try:
        props = AllChem.MMFFGetMoleculeProperties(mol3d)
        ff = AllChem.MMFFGetMoleculeForceField(mol3d, props, confId=cid)
        ff.Minimize()
        energy = ff.CalcEnergy()
    except:
        ff = AllChem.UFFGetMoleculeForceField(mol3d, confId=cid)
        ff.Minimize()
        energy = ff.CalcEnergy()

    energies.append((cid, energy))

print("")

print("Minimization complete.")

for cid, e in energies:
    print(f"Conformer {cid}: Energy = {e:.4f}")

Minimizing conformers...

Minimization complete.
Conformer 0: Energy = -122.5284
Conformer 1: Energy = -122.5284
Conformer 2: Energy = -122.5284
Conformer 3: Energy = -122.5284
Conformer 4: Energy = -122.5284
Conformer 5: Energy = -122.5284
Conformer 6: Energy = -122.5284
Conformer 7: Energy = -122.5284
Conformer 8: Energy = -122.5284
Conformer 9: Energy = -122.5284
Conformer 10: Energy = -122.5284
Conformer 11: Energy = -122.5284
Conformer 12: Energy = -122.5284
Conformer 13: Energy = -122.5284
Conformer 14: Energy = -122.5284
Conformer 15: Energy = -122.5284
Conformer 16: Energy = -122.5284
Conformer 17: Energy = -122.5284
Conformer 18: Energy = -122.5284
Conformer 19: Energy = -122.5284


Explanation:

Multiple 3D conformers are generated to represent different possible geometries.

Energy minimization ensures stable conformations using MMFF first, then UFF if MMFF is unavailable.

## 3️⃣ Score Conformers and Filter by Energy

In [6]:
import numpy as np

energies_array = np.array([e for _, e in energies])
min_energy = energies_array.min()
energy_window = 5.0

good_conformers = []

for cid, energy in energies:
    if energy <= min_energy + energy_window:
        good_conformers.append((cid, energy))

good_conformers

[(0, -122.52841057522076),
 (1, -122.52841057495584),
 (2, -122.52841057516962),
 (3, -122.52841057522609),
 (4, -122.52841057528585),
 (5, -122.52841057464775),
 (6, -122.52841057573947),
 (7, -122.52841057564356),
 (8, -122.52841057517732),
 (9, -122.52841057456651),
 (10, -122.52841057432848),
 (11, -122.52841057375932),
 (12, -122.52841057470653),
 (13, -122.52841057564689),
 (14, -122.52841057542395),
 (15, -122.52841057530439),
 (16, -122.52841057492422),
 (17, -122.52841057552074),
 (18, -122.52841057547326),
 (19, -122.52841057473037)]

Explanation:

Calculates energy for each conformer.

Filters low-energy conformers for further pharmacophore analysis.
low-energy conformers (Stable)

## 4️⃣ Extract Pharmacophore Features

In [7]:
from rdkit.Chem import ChemicalFeatures
from rdkit import RDConfig
import os

fdef_path = os.path.join(RDConfig.RDDataDir, 'BaseFeatures.fdef')
factory = ChemicalFeatures.BuildFeatureFactory(fdef_path)

all_features = {}

for cid, energy in good_conformers:
    feats = factory.GetFeaturesForMol(mol3d, confId=cid)
    all_features[cid] = feats

all_features

{0: (<rdkit.Chem.rdMolChemicalFeatures.MolChemicalFeature at 0x781397a07e60>,
  <rdkit.Chem.rdMolChemicalFeatures.MolChemicalFeature at 0x78138ed05540>),
 1: (<rdkit.Chem.rdMolChemicalFeatures.MolChemicalFeature at 0x78138ed055b0>,
  <rdkit.Chem.rdMolChemicalFeatures.MolChemicalFeature at 0x78138ed057e0>),
 2: (<rdkit.Chem.rdMolChemicalFeatures.MolChemicalFeature at 0x78138ed05230>,
  <rdkit.Chem.rdMolChemicalFeatures.MolChemicalFeature at 0x78138ed05a80>),
 3: (<rdkit.Chem.rdMolChemicalFeatures.MolChemicalFeature at 0x78138ed05af0>,
  <rdkit.Chem.rdMolChemicalFeatures.MolChemicalFeature at 0x78138ed05d20>),
 4: (<rdkit.Chem.rdMolChemicalFeatures.MolChemicalFeature at 0x78138ed05d90>,
  <rdkit.Chem.rdMolChemicalFeatures.MolChemicalFeature at 0x78138ed05fc0>),
 5: (<rdkit.Chem.rdMolChemicalFeatures.MolChemicalFeature at 0x78138ed06030>,
  <rdkit.Chem.rdMolChemicalFeatures.MolChemicalFeature at 0x78138ed06260>),
 6: (<rdkit.Chem.rdMolChemicalFeatures.MolChemicalFeature at 0x78138ed062d0>

Explanation:

Extracts pharmacophore features like hydrogen bond donors, acceptors, and aromatic rings using RDKit’s predefined features.

## 5️⃣ Compute Feature Coordinates

In [8]:
from rdkit.Geometry import Point3D

def get_feature_point(feature, mol, cid):
    atom_ids = feature.GetAtomIds()
    conf = mol.GetConformer(cid)

    if len(atom_ids) == 1:
        idx = atom_ids[0]
        pos = conf.GetAtomPosition(idx)
        return (pos.x, pos.y, pos.z)

    xs = []
    ys = []
    zs = []
    for idx in atom_ids:
        pos = conf.GetAtomPosition(idx)
        xs.append(pos.x)
        ys.append(pos.y)
        zs.append(pos.z)

    return (sum(xs)/len(xs), sum(ys)/len(ys), sum(zs)/len(zs))


feature_coords = {}

for cid, feats in all_features.items():
    coords = []
    for f in feats:
        coords.append({
            "type": f.GetType(),
            "point": get_feature_point(f, mol3d, cid)
        })
    feature_coords[cid] = coords

feature_coords

{0: [{'type': 'SingleAtomDonor',
   'point': (-1.31270280531742, 1.126339020106633, 0.21294693304186232)},
  {'type': 'SingleAtomAcceptor',
   'point': (0.5996386634429802, 2.4231214907069134, 0.4704481994242097)},
  {'type': 'SingleAtomAcceptor',
   'point': (-3.1804563425331183, -0.1954407018195363, -0.04922830340580211)},
  {'type': 'SingleAtomAcceptor',
   'point': (1.1633381311901139, -2.041645665347401, -0.3903737143734171)},
  {'type': 'Imidazole',
   'point': (1.3385468279029384, -0.8936347020904083, -0.16788025070372722)},
  {'type': 'Arom5',
   'point': (1.3385468279029384, -0.8936347020904083, -0.16788025070372722)}],
 1: [{'type': 'SingleAtomDonor',
   'point': (-1.2181570867861913, 1.2392872404576776, -0.2049004546763943)},
  {'type': 'SingleAtomAcceptor',
   'point': (0.7921635835416546, 2.3940403100724037, -0.3785617789333029)},
  {'type': 'SingleAtomAcceptor',
   'point': (-3.1860373788826406, 0.05580079786956699, -0.028411796627263397)},
  {'type': 'SingleAtomAcceptor'

Explanation:

Coordinates of pharmacophore features are calculated for visualization and clustering.

## 6️⃣ Compute Pairwise Feature Distances

In [9]:
import math

def distance(p1, p2):
    return math.sqrt(
        (p1[0] - p2[0])**2 +
        (p1[1] - p2[1])**2 +
        (p1[2] - p2[2])**2
    )

max_cutoff = 7.0

pairwise_distances = {}

for cid, feats in feature_coords.items():
    dist_list = []
    for i in range(len(feats)):
        for j in range(i+1, len(feats)):
            p1 = feats[i]["point"]
            p2 = feats[j]["point"]
            d = distance(p1, p2)
            if d <= max_cutoff:
                dist_list.append({
                    "feat1": feats[i]["type"],
                    "feat2": feats[j]["type"],
                    "distance": d
                })
    pairwise_distances[cid] = dist_list

pairwise_distances

{0: [{'feat1': 'SingleAtomDonor',
   'feat2': 'SingleAtomAcceptor',
   'distance': 2.3248659254643833},
  {'feat1': 'SingleAtomDonor',
   'feat2': 'SingleAtomAcceptor',
   'distance': 2.303115447319861},
  {'feat1': 'SingleAtomDonor',
   'feat2': 'SingleAtomAcceptor',
   'distance': 4.065821133563305},
  {'feat1': 'SingleAtomDonor',
   'feat2': 'Imidazole',
   'distance': 3.354764939589248},
  {'feat1': 'SingleAtomDonor',
   'feat2': 'Arom5',
   'distance': 3.354764939589248},
  {'feat1': 'SingleAtomAcceptor',
   'feat2': 'SingleAtomAcceptor',
   'distance': 4.627747819180297},
  {'feat1': 'SingleAtomAcceptor',
   'feat2': 'SingleAtomAcceptor',
   'distance': 4.581802834583122},
  {'feat1': 'SingleAtomAcceptor',
   'feat2': 'Imidazole',
   'distance': 3.457502006970886},
  {'feat1': 'SingleAtomAcceptor',
   'feat2': 'Arom5',
   'distance': 3.457502006970886},
  {'feat1': 'SingleAtomAcceptor',
   'feat2': 'SingleAtomAcceptor',
   'distance': 4.732166880904316},
  {'feat1': 'SingleAtomAc

Explanation:

Calculates spatial relationships between features using Euclidean distance.

## 7️⃣ Build Consensus Pharmacophore

In [10]:
import math

def dist(p1, p2):
    return math.sqrt(
        (p1[0]-p2[0])**2 +
        (p1[1]-p2[1])**2 +
        (p1[2]-p2[2])**2
    )

same_type_clusters = {}
threshold = 1.5

for cid, feats in feature_coords.items():
    for feat in feats:
        ftype = feat["type"]
        point = feat["point"]

        if ftype not in same_type_clusters:
            same_type_clusters[ftype] = []

        placed = False
        for cluster in same_type_clusters[ftype]:
            if dist(point, cluster["center"]) < threshold:
                cluster["points"].append(point)
                xs = [p[0] for p in cluster["points"]]
                ys = [p[1] for p in cluster["points"]]
                zs = [p[2] for p in cluster["points"]]
                cluster["center"] = (
                    sum(xs)/len(xs),
                    sum(ys)/len(ys),
                    sum(zs)/len(zs)
                )
                placed = True
                break

        if not placed:
            same_type_clusters[ftype].append({
                "points": [point],
                "center": point
            })

consensus_pharmacophore = {}

for ftype, clusters in same_type_clusters.items():
    consensus_pharmacophore[ftype] = [
        cluster["center"] for cluster in clusters
    ]

consensus_pharmacophore

{'SingleAtomDonor': [(-1.062771384571464,
   1.3376327326726305,
   -0.0751342777082694),
  (-1.3282120419123302, -1.100483323328049, -0.020122534738302455),
  (1.5445659820165993, -0.7591436237055151, 0.2742120955832425)],
 'SingleAtomAcceptor': [(1.1037467614950374,
   2.165679948302828,
   -0.07830255927427399),
  (-3.1300525874269054, 0.33370310972008843, -0.025680284665006665),
  (0.6248845022444525, -2.311746152376298, 0.07007050586318472),
  (3.041290898659546, 0.8131448539907006, -0.49524603821424834),
  (-1.6189227002227715, 1.6191942244865434, -0.6567810468962378)],
 'Imidazole': [(1.13108375691517, -1.114197921607843, 0.017645315380625356),
  (1.3480616419998197, 0.8656423394550851, -0.038392105062091626),
  (-1.5138834915821258, 0.5429120940612104, -0.1785408166683937)],
 'Arom5': [(1.13108375691517, -1.114197921607843, 0.017645315380625356),
  (1.3480616419998197, 0.8656423394550851, -0.038392105062091626),
  (-1.5138834915821258, 0.5429120940612104, -0.1785408166683937)]}

Explanation:

Consolidates features by type across all conformers to form a consensus pharmacophore.

## 8️⃣ Export to JSON

In [11]:
mol_name = "Conformer3D_COMPOUND_CID_2519"
import json

output = {
    "molecule": mol_name,
    "consensus_pharmacophore": consensus_pharmacophore
}

with open("pharmacophore_output.json", "w") as f:
    json.dump(output, f, indent=4)

"JSON file saved as pharmacophore_output.json"

'JSON file saved as pharmacophore_output.json'

Explanation:

Saves all pharmacophore data in JSON format for downstream analysis.